# Statistika podatkov za diplomo (Colab)

Zvezek **ne spreminja nobenih podatkov**. Iz datotek na Drive izračuna številke za poglavji Materiali in metode ter Rezultati in jih shrani kot majhne datoteke v `MyDrive/Diploma/diploma_statistika/`:

| Datoteka | Za kaj |
|---|---|
| `verzije.json` | verzije Pythona, paketov in GPU (razdelek 3.8) |
| `tabela1_celice_po_bolnikih.csv` | tabela 1: celice po bolniku in kvadrantu, kloni, ekspandirane celice, merilo izbire |
| `povzetek_podatkov.json` | skupna števila celic, klonov, CD4/CD8 |
| `qc_po_datotekah.csv` | koliko celic odstrani vsak korak kontrole kakovosti (po datoteki .h5) |
| `qc_hist_geni.csv`, `qc_hist_mito.csv` | podatki za sliko 5 (porazdelitev števila genov in deleža mitohondrijskih prepisov) |

**Kako poženeš:** Runtime → Run all. Za razdelek 3 (surove datoteke .h5) izberi High-RAM, ker naloži vse surove matrike kot zvezek 01. Če želiš samo tabelo 1, nastavi `RUN_QC_RAW = False`.

Na koncu mi samo sporoči, da je končano. Datoteke preberem z Drive; če to ne gre, jih naloži v nit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, sys, glob, re, platform, importlib
import numpy as np
import pandas as pd

# Poti enake kot v notebooks/01_preprocessing.ipynb
RAW_DIR  = '/content/drive/MyDrive/Diploma/data/GSE200996_RAW'
META_DIR = '/content/drive/MyDrive/Diploma/data'
PROC_DIR = '/content/drive/MyDrive/Diploma/data/processed'
OUT_DIR  = '/content/drive/MyDrive/Diploma/diploma_statistika'

MIN_EXP     = 20     # merilo izbire izpuščenih bolnikov (notebooks_flux/00_diagnostika_pacientov.ipynb)
MIN_GENES   = 200    # kontrola kakovosti (01_preprocessing)
MAX_MITO    = 20     # %
RUN_QC_RAW  = True   # razdelek 3: naloži surove .h5 (High-RAM)

os.makedirs(OUT_DIR, exist_ok=True)
print('Izhod:', OUT_DIR)

## 1. Verzije programske opreme

In [ ]:
def ver(mod):
    try:
        m = importlib.import_module(mod)
        return getattr(m, '__version__', 'ni podatka')
    except Exception as e:
        return 'ni nameščen'

paketi = ['numpy', 'pandas', 'scipy', 'scanpy', 'anndata', 'torch', 'sklearn', 'umap', 'matplotlib', 'seaborn', 'cobra']
verzije = {'python': platform.python_version(), 'platforma': platform.platform()}
verzije.update({p: ver(p) for p in paketi})
try:
    import torch
    verzije['cuda'] = torch.version.cuda
    verzije['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ni GPU'
except Exception:
    verzije['gpu'] = 'ni podatka'
with open(os.path.join(OUT_DIR, 'verzije.json'), 'w') as f:
    json.dump(verzije, f, indent=2, ensure_ascii=False)
print(json.dumps(verzije, indent=2, ensure_ascii=False))

## 2. Tabela 1: celice po bolniku in kvadrantu

Kvadrant = tkivo (kri, tumor) × faza (pred, po zdravljenju). Ekspanzija je definirana enako kot pri vrednotenju: klon je ekspandiran, če ima v krvi po zdravljenju več celic kot pred njim (`df_all_tcrs`, stolpca 1 in 0). `bolnik_indeks` je številka, ki jo uporabljajo logi (`heldout_patient`), `bolnik_id` pa oznaka iz GEO.

In [ ]:
labels     = pd.read_pickle(os.path.join(PROC_DIR, 'data_labels.pkl'))
labels_str = pd.read_pickle(os.path.join(PROC_DIR, 'data_labels_str.pkl'))
tcrs       = pd.read_pickle(os.path.join(PROC_DIR, 'df_all_tcrs.pkl'))

assert len(labels) == len(labels_str), 'data_labels in data_labels_str nista poravnana'
L = labels.reset_index(drop=True)
S = labels_str.reset_index(drop=True)

tissue  = L['Tissue'].astype(int).values            # 0 kri, 1 tumor
stage   = L['Treatment Stage'].astype(int).values   # 0 pred, 1 po
patient = L['Patient'].astype(int).values
ctype   = L['SubCellType'].astype(int).values       # 0 CD4, 1 CD8
tcr_idx = L['CDR3(Beta1)'].astype(int).values

blood_expanded = (tcrs.iloc[:, 1].fillna(0) > tcrs.iloc[:, 0].fillna(0)).values

rows = []
for pid in sorted(np.unique(patient)):
    m = patient == pid
    ids = S.loc[m, 'Patient'].unique()
    r = {'bolnik_indeks': int(pid), 'bolnik_id': ','.join(map(str, ids)), 'celic': int(m.sum()),
         'CD4': int((m & (ctype == 0)).sum()), 'CD8': int((m & (ctype == 1)).sum())}
    for t, tn in [(0, 'kri'), (1, 'tumor')]:
        for s, sn in [(0, 'pred'), (1, 'po')]:
            r[f'{tn}_{sn}'] = int((m & (tissue == t) & (stage == s)).sum())
    r['klonov'] = int(len(np.unique(tcr_idx[m])))
    bp = m & (tissue == 0) & (stage == 0)
    r['ekspandiranih_celic_kri_pred'] = int(blood_expanded[tcr_idx[bp]].sum()) if bp.any() else 0
    r['ekspandiranih_klonov_kri_pred'] = int(len(np.unique(tcr_idx[bp][blood_expanded[tcr_idx[bp]]]))) if bp.any() else 0
    r['izpolnjuje_merilo'] = r['ekspandiranih_celic_kri_pred'] >= MIN_EXP
    rows.append(r)

tab1 = pd.DataFrame(rows)
tab1.to_csv(os.path.join(OUT_DIR, 'tabela1_celice_po_bolnikih.csv'), index=False)

povzetek = {
    'celic': int(len(L)), 'bolnikov': int(len(tab1)),
    'CD4': int((ctype == 0).sum()), 'CD8': int((ctype == 1).sum()),
    'razlicnih_CDR3b': int(len(np.unique(tcr_idx))), 'vrstic_df_all_tcrs': int(len(tcrs)),
    'kri_pred': int(((tissue == 0) & (stage == 0)).sum()), 'kri_po': int(((tissue == 0) & (stage == 1)).sum()),
    'tumor_pred': int(((tissue == 1) & (stage == 0)).sum()), 'tumor_po': int(((tissue == 1) & (stage == 1)).sum()),
    'bolnikov_ki_izpolnjujejo_merilo': int(tab1['izpolnjuje_merilo'].sum()),
    'merilo': f'najmanj {MIN_EXP} ekspandiranih celic v krvi pred zdravljenjem',
}
with open(os.path.join(OUT_DIR, 'povzetek_podatkov.json'), 'w') as f:
    json.dump(povzetek, f, indent=2, ensure_ascii=False)

pd.set_option('display.max_rows', 100, 'display.width', 200)
print(json.dumps(povzetek, indent=2, ensure_ascii=False))
print(tab1.sort_values('ekspandiranih_celic_kri_pred', ascending=False).to_string(index=False))

## 3. Kontrola kakovosti na surovih datotekah .h5 (za sliko 5 in Metode)

Ponovi korake iz `01_preprocessing` (najmanj 200 genov, ujemanje s celicami iz metapodatkov, delež mitohondrijskih prepisov pod 20 %) in prešteje, koliko celic odstrani vsak korak. Za histograme uporabi celice, ki se ujemajo z metapodatki, **pred** filtriranjem po številu genov.

In [ ]:
if RUN_QC_RAW:
    import scanpy as sc, anndata

    def load_meta(fp):
        return pd.read_csv(fp, sep='\t', index_col=0)
    parts = []
    for fn, cls, tis in [('GSE200996_CD4.PBMC.single.cell.meta.data.txt', 'CD4', 'Blood'),
                         ('GSE200996_CD8.PBMC.single.cell.meta.data.txt', 'CD8', 'Blood'),
                         ('GSE200996_CD4.tumor.single.cell.meta.data.txt', 'CD4', 'Tumor'),
                         ('GSE200996_CD8.tumor.single.cell.meta.data.txt', 'CD8', 'Tumor')]:
        d = load_meta(os.path.join(META_DIR, fn)); d['CellClass'] = cls; d['Tissue_str'] = tis
        parts.append(d)
    meta_all = pd.concat(parts)
    meta_all = meta_all[meta_all['Stage'] != 'B3']
    meta_all['barcode_clean'] = meta_all.index.str.split('_').str[0]
    print('Celic v metapodatkih (brez B3):', len(meta_all))

    def patients_from_filename(fn):
        mt = re.search(r'(P\d{2}(?:-P\d{2})*)', os.path.basename(fn))
        return mt.group(1).split('-') if mt else None

    qc_rows, genes_all, mito_all = [], [], []
    for pattern, tis in [('*_GEX_sc_PBMC.h5', 'kri'), ('*_GEX_sc_tumor.h5', 'tumor')]:
        for f in sorted(glob.glob(os.path.join(RAW_DIR, pattern))):
            pats = patients_from_filename(f)
            if pats is None:
                continue
            valid_bc = set(meta_all.loc[meta_all['Patient_ID'].isin(pats), 'barcode_clean'])
            ad = sc.read_10x_h5(f); ad.var_names_make_unique()
            X = ad.X.tocsr()
            n_genes = np.asarray((X > 0).sum(axis=1)).ravel()
            total = np.asarray(X.sum(axis=1)).ravel()
            mt = ad.var_names.str.upper().str.startswith('MT-')
            mito_pct = 100 * np.asarray(X[:, np.where(mt)[0]].sum(axis=1)).ravel() / np.maximum(total, 1)
            bc = ad.obs_names.str.replace(r'-\d+$', '', regex=True)
            in_meta = np.asarray(bc.isin(valid_bc))
            ok_genes = n_genes >= MIN_GENES
            ok_mito = mito_pct < MAX_MITO
            qc_rows.append({'datoteka': os.path.basename(f), 'tkivo': tis, 'bolniki': '-'.join(pats),
                            'surove_crtne_kode': int(len(n_genes)),
                            'genov_vsaj_200': int(ok_genes.sum()),
                            'po_ujemanju_z_metapodatki': int((ok_genes & in_meta).sum()),
                            'po_mito_pod_20': int((ok_genes & in_meta & ok_mito).sum()),
                            'odstranjenih_zaradi_mito': int((ok_genes & in_meta & ~ok_mito).sum())})
            genes_all.append(n_genes[in_meta]); mito_all.append(mito_pct[in_meta & ok_genes])
            print(qc_rows[-1])
            del ad, X

    qc = pd.DataFrame(qc_rows)
    qc.to_csv(os.path.join(OUT_DIR, 'qc_po_datotekah.csv'), index=False)
    print(qc.sum(numeric_only=True))

    g = np.concatenate(genes_all); mpc = np.concatenate(mito_all)
    edges_g = np.linspace(0, 4.5, 91)                       # log10(genov + 1)
    hg, _ = np.histogram(np.log10(g + 1), bins=edges_g)
    pd.DataFrame({'log10_genov_od': edges_g[:-1], 'log10_genov_do': edges_g[1:], 'celic': hg}).to_csv(
        os.path.join(OUT_DIR, 'qc_hist_geni.csv'), index=False)
    edges_m = np.arange(0, 101, 1.0)
    hm, _ = np.histogram(np.clip(mpc, 0, 100), bins=edges_m)
    pd.DataFrame({'mito_od': edges_m[:-1], 'mito_do': edges_m[1:], 'celic': hm}).to_csv(
        os.path.join(OUT_DIR, 'qc_hist_mito.csv'), index=False)
    print('Celic, ujemajočih se z metapodatki (pred QC):', len(g), '| pod 200 genov:', int((g < MIN_GENES).sum()),
          '| mediana mito % (po pragu genov):', round(float(np.median(mpc)), 2))
else:
    print('Razdelek 3 preskočen (RUN_QC_RAW = False).')

## 4. Kaj je shranjeno

In [ ]:
for fn in sorted(os.listdir(OUT_DIR)):
    print(f'{fn:40s} {os.path.getsize(os.path.join(OUT_DIR, fn)):>8d} B')